In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

# 1. 경로 설정
# 현재 파일 위치 기준: 02_code/04_data_labeling.py
BASE_DIR = Path(os.getcwd()).parent
PROCESSED_DIR = BASE_DIR / "01_data" / "02_processed"
FEATURE_PATH = PROCESSED_DIR / "02_sensor_daily_features.csv"
SAVE_PATH = PROCESSED_DIR / "03_sensor_labeled_data.csv"

# 2. 데이터 로드
if not FEATURE_PATH.exists():
    print(f"❌ 파일을 찾을 수 없습니다: {FEATURE_PATH}")
    exit()

df = pd.read_csv(FEATURE_PATH)
print(f"📂 분석용 지표 데이터 로드 완료: {len(df):,}행")

# 3. 통계적 임계치(Threshold) 산출
# [위험 2] 기준: 상위 25% (Q3)
q3_corr = df['avg_corr'].quantile(0.75)
q3_peak = df['daily_peak'].quantile(0.75)
q3_ratio = df['peak_ratio'].quantile(0.75)

# [주의 1] 기준: 중앙값 (Median)
med_corr = df['avg_corr'].median()
med_peak = df['daily_peak'].median()
med_ratio = df['peak_ratio'].median()

print(f"\n📊 [산출된 임계치 기준]")
print(f"- 위험(Q3): Corr >= {q3_corr:.3f}, Peak >= {q3_peak}, Ratio >= {q3_ratio:.2f}")
print(f"- 주의(Med): Corr >= {med_corr:.3f}, Peak >= {med_peak}, Ratio >= {med_ratio:.2f}")

# 4. 삼중 분류 라벨링 함수 정의
def final_labeling(row):
    """3단계 로직 기반 삼중 분류 (0:정상, 1:주의, 2:위험)"""
    
    # [위험 2] : 모든 핵심 지표가 상위 25% 이내 (교집합)
    is_danger = (row['avg_corr'] >= q3_corr) and \
                (row['daily_peak'] >= q3_peak) and \
                (row['peak_ratio'] >= q3_ratio)
    
    if is_danger:
        return 2
    
    # [주의 1] : 위험은 아니지만, 모든 지표가 중앙값(평균적 수준) 이상인 경우
    is_caution = (row['avg_corr'] >= med_corr) and \
                 (row['daily_peak'] >= med_peak) and \
                 (row['peak_ratio'] >= med_ratio)
    
    if is_caution:
        return 1
    
    # [정상 0] : 그 외 모든 경우 (소리가 작거나 불규칙함)
    return 0

# 5. 라벨 적용 및 결과 확인
print("\n🏷️ 라벨링 작업 진행 중...")
df['label'] = df.apply(final_labeling, axis=1)

# 분포 확인
dist = df['label'].value_counts(normalize=True).sort_index() * 100
counts = df['label'].value_counts().sort_index()

print("\n✅ 라벨링 분포 결과:")
for label, pct in dist.items():
    name = {0: "정상(0)", 1: "주의(1)", 2: "위험(2)"}[label]
    print(f"- {name}: {counts[label]:>5,}개 ({pct:.2f}%)")

# 6. 결과 저장
df.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')
print(f"\n💾 최종 라벨링 데이터 저장 완료: {SAVE_PATH}")


📂 분석용 지표 데이터 로드 완료: 15,833행

📊 [산출된 임계치 기준]
- 위험(Q3): Corr >= 0.914, Peak >= 30.0, Ratio >= 79.58
- 주의(Med): Corr >= 0.871, Peak >= 16.0, Ratio >= 66.94

🏷️ 라벨링 작업 진행 중...

✅ 라벨링 분포 결과:
- 정상(0): 12,780개 (80.72%)
- 주의(1): 2,068개 (13.06%)
- 위험(2):   985개 (6.22%)

💾 최종 라벨링 데이터 저장 완료: c:\workspace\wvr\classification\labeling_project\01_data\02_processed\sensor_labeled_data.csv
